In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [ ]:
!pip install -q gwosc gwpy pycbc


In [ ]:
import os
import json
import math
import time
import hashlib
import argparse
import warnings
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import h5py

In [ ]:
@dataclass
class Config:
    # sampling / duration 
    detector: str = "H1"
    sample_rate: int = 4096          # Hz
    duration: float = 1.0            # seconds
    n_samples: int = 4096            # sample_rate * duration

    # templates
    n_templates: int = 80
    n_examples_per_template: int = 50
    # For a one-second dataset, these BBH masses retain a useful merger chirp
    # while keeping a complete waveform practical at 4096 Hz.
    mass_min: float = 20.0
    mass_max: float = 75.0
    max_mass_ratio: float = 8.0
    # Binary classification only 
    # bank varies component masses / mass ratio only. No spin, eccentricity,
    approximant: str = "IMRPhenomD"
    f_lower_start: float = 50.0
    f_lower_max: float = 120.0
    f_lower_step: float = 10.0
    max_waveform_fraction: float = 0.75   # retain the merger-containing final
                                           # 0.75 s, leaving placement jitter

    # --- SNR distribution (mixture) ---
    snr_bins: Tuple[Tuple[float, float, float], ...] = (
        (0.55, 8.0, 14.0),   # (weight, low, high)
        (0.30, 14.0, 20.0),
        (0.15, 20.0, 30.0),
    )

    # --- template-disjoint split ---
    n_train_templates: int = 56
    n_val_templates: int = 12
    n_test_templates: int = 12

    # --- noise-only class ---
    n_noise_only_total: int = 1600  # split proportionally to template split (70/15/15)
    real_event_names: Dict[str, List[str]] = field(default_factory=lambda: {
        "train": ["GW150914", "GW151226", "GW170104", "GW170608", "GW170729", "GW170809"],
        "val":   ["GW170814", "GW170818"],
        "test":  ["GW170817", "GW170823"],
    })

    # --- GWOSC noise fetching ---
    noise_time_ranges: Dict[str, List[Tuple[int, int]]] = field(default_factory=lambda: {
        "train": [(1238166018, 1238169618), (1238180000, 1238183000)],  # ~1h40m total
        "val":   [(1238200000, 1238201800)],                             # ~30 min
        "test":  [(1238210000, 1238211800)],                             # ~30 min
    })
    known_event_gps: Tuple[float, ...] = (1238166018.0 + 1817.0,) 
    event_exclusion_buffer: float = 16.0  # seconds

    # --- output ---
    output_dir: str = "gw_dataset_output"
    hdf5_filename: str = "gw_dataset.h5"
    metadata_filename: str = "metadata.csv"
    random_seed: int = 42

    def __post_init__(self):
        self.n_samples = int(self.sample_rate * self.duration)
        assert self.n_train_templates + self.n_val_templates + self.n_test_templates == self.n_templates



In [ ]:
@dataclass
class TemplateParams:
    template_id: str
    mass1: float
    mass2: float
    f_lower: float


def physical_waveform_support(hp, sample_rate: int) -> np.ndarray:
    arr = np.asarray(hp.numpy() if hasattr(hp, "numpy") else hp, dtype=np.float64)
    peak = np.max(np.abs(arr))
    if not np.isfinite(peak) or peak == 0.0:
        raise RuntimeError("Waveform has no finite nonzero samples.")
    support = np.flatnonzero(np.abs(arr) >= peak * 1e-5)
    if len(support) == 0:
        raise RuntimeError("Could not locate physical waveform support.")
    pad = max(16, int(0.02 * sample_rate))
    start = max(0, int(support[0]) - pad)
    end = min(len(arr), int(support[-1]) + pad + 1)
    return arr[start:end].copy()


def generate_template_bank(cfg: Config, rng: np.random.Generator) -> List[TemplateParams]:
    
    templates = []
    seen_keys = set()
    attempts = 0
    while len(templates) < cfg.n_templates and attempts < cfg.n_templates * 50:
        attempts += 1
        m1 = rng.uniform(cfg.mass_min, cfg.mass_max)
        m2 = rng.uniform(cfg.mass_min, cfg.mass_max)
        if m1 < m2:
            m1, m2 = m2, m1
        if m1 / m2 > cfg.max_mass_ratio:
            continue

        # avoid near-duplicate templates
        key = (round(m1, 1), round(m2, 1))
        if key in seen_keys:
            continue
        seen_keys.add(key)

        tid = f"tpl_{len(templates):03d}"
        candidate = TemplateParams(
            template_id=tid, mass1=m1, mass2=m2, f_lower=cfg.f_lower_start,
        )
        templates.append(candidate)

    if len(templates) < cfg.n_templates:
        raise RuntimeError("Could not sample enough distinct templates; loosen constraints.")
    return templates


def fit_f_lower(template: TemplateParams, cfg: Config) -> TemplateParams:
    from pycbc.waveform import get_td_waveform  # lazy import

    f_lower = cfg.f_lower_start
    max_len_samples = int(cfg.max_waveform_fraction * cfg.n_samples)

    while f_lower <= cfg.f_lower_max:
        try:
            hp, _ = get_td_waveform(
                approximant=cfg.approximant,
                mass1=template.mass1, mass2=template.mass2,
                delta_t=1.0 / cfg.sample_rate, f_lower=f_lower,
            )
        except Exception:
            f_lower += cfg.f_lower_step
            continue

        hp_arr = physical_waveform_support(hp, cfg.sample_rate)
        if len(hp_arr) <= max_len_samples:
            template.f_lower = f_lower
            return template
        f_lower += cfg.f_lower_step

    template.f_lower = cfg.f_lower_max
    return template



# TEMPLATE-DISJOINT SPLIT

def split_templates(templates: List[TemplateParams], cfg: Config,
                     rng: np.random.Generator) -> Dict[str, List[TemplateParams]]:
    idx = rng.permutation(len(templates))
    train_idx = idx[:cfg.n_train_templates]
    val_idx = idx[cfg.n_train_templates: cfg.n_train_templates + cfg.n_val_templates]
    test_idx = idx[cfg.n_train_templates + cfg.n_val_templates:]

    splits = {
        "train": [templates[i] for i in train_idx],
        "val": [templates[i] for i in val_idx],
        "test": [templates[i] for i in test_idx],
    }

    #template IDs must be disjoint across splits
    ids = {k: set(t.template_id for t in v) for k, v in splits.items()}
    assert ids["train"].isdisjoint(ids["val"])
    assert ids["train"].isdisjoint(ids["test"])
    assert ids["val"].isdisjoint(ids["test"])
    return splits


In [ ]:
def auto_select_noise_time_ranges(
    run_start: int = 1238166018,   # O3a start
    run_end: int = 1253977218,     # O3a end
    detector: str = "H1",
    train_hours: float = 2.0,
    val_hours: float = 0.75,
    test_hours: float = 0.75,
    min_segment_seconds: int = 600,
) -> Tuple[Dict[str, List[Tuple[int, int]]], Tuple[float, ...]]:
    from gwosc.timeline import get_segments
    from gwosc.datasets import find_datasets, event_gps

    total_span = run_end - run_start
    # three non-overlapping candidate windows: early / mid / late in the run
    train_window = (
    run_start + int(0.05 * total_span),
    run_start + int(0.05 * total_span)
    + int(train_hours * 3600)
     )

    val_window = (
    run_start + int(0.45 * total_span),
    run_start + int(0.45 * total_span)
    + int(val_hours * 3600)
     )

    test_window = (
    run_start + int(0.85 * total_span),
    run_start + int(0.85 * total_span)
    + int(test_hours * 3600)
     )

    candidate_windows = {"train": train_window, "val": val_window, "test": test_window}
    noise_time_ranges: Dict[str, List[Tuple[int, int]]] = {}
    all_event_gps: List[float] = []

    flag = f"{detector}_DATA"  # science-quality "detector is taking data" flag

    for split, (w_start, w_end) in candidate_windows.items():
        try:
            segs = get_segments(flag, w_start, w_end)
        except Exception as e:
            raise RuntimeError(
                f"Could not query GWOSC timeline for split={split} "
                f"({w_start}, {w_end}): {e}. Check internet access."
            )
        good_segs = [(s, e) for (s, e) in segs if (e - s) >= min_segment_seconds]
        if not good_segs:
            raise RuntimeError(
                f"No science-quality segments >= {min_segment_seconds}s found "
                f"for split={split} in window ({w_start}, {w_end}). "
                f"Try a different run_start/run_end or lower min_segment_seconds."
            )
        noise_time_ranges[split] = good_segs

        # find any real events inside this window, to auto-exclude them
        try:
            events_in_window = find_datasets(
                type="event", detector=detector, segment=(w_start, w_end)
            )
            for ev in events_in_window:
                try:
                    all_event_gps.append(float(event_gps(ev)))
                except Exception:
                    pass
        except Exception:
            pass  

    known_event_gps = tuple(sorted(set(all_event_gps)))
    return noise_time_ranges, known_event_gps



def _fetch_open_data_with_retry(TimeSeries, detector, gps_start, gps_end, sample_rate,
                                 max_retries: int = 4, base_delay: float = 5.0):
    import time as _time
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return TimeSeries.fetch_open_data(
                detector, gps_start, gps_end, sample_rate=sample_rate, verbose=False
            )
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                delay = base_delay * (2 ** (attempt - 1))
                print(f"      GWOSC fetch failed (attempt {attempt}/{max_retries}): {e}. "
                      f"Retrying in {delay:.0f}s...")
                _time.sleep(delay)
    raise RuntimeError(
        f"Failed to fetch {detector} data for ({gps_start}, {gps_end}) after "
        f"{max_retries} attempts. Last error: {last_err}"
    )


def fetch_noise_segments(cfg: Config, split: str) -> List[Dict]:
    from gwpy.timeseries import TimeSeries  # lazy import

    segments = []
    for (gps_start, gps_end) in cfg.noise_time_ranges[split]:
        ts = _fetch_open_data_with_retry(
            TimeSeries, cfg.detector, gps_start, gps_end, cfg.sample_rate
        )
        arr = ts.value.astype(np.float32)
        n_win = len(arr) // cfg.n_samples
        for i in range(n_win):
            win_start_gps = gps_start + i * cfg.duration
            win_end_gps = win_start_gps + cfg.duration

            # exclude windows overlapping a known event
            for ev in cfg.known_event_gps:
               if (win_start_gps - cfg.event_exclusion_buffer
            <= ev
            <= win_end_gps + cfg.event_exclusion_buffer):
                   break
            else:
                chunk = arr[i * cfg.n_samples:(i + 1) * cfg.n_samples]

                if len(chunk) != cfg.n_samples:
                    continue

                if not np.all(np.isfinite(chunk)):
                   continue

                seg_id = f"{cfg.detector}_{split}_{int(win_start_gps)}"

                segments.append({
        "noise_segment_id": seg_id,
        "data": chunk,
        "gps_start": win_start_gps,
        "raw_ts": ts,
    })

    return segments


def fetch_real_event_segments(cfg: Config, split: str) -> List[Dict]:
   
    from gwosc.datasets import event_gps
    from gwpy.timeseries import TimeSeries

    events = []
    half_window = cfg.duration / 2.0
    for event_name in cfg.real_event_names.get(split, []):
        try:
            gps = float(event_gps(event_name))
            ts = _fetch_open_data_with_retry(
                TimeSeries, cfg.detector, gps - half_window, gps + half_window,
                cfg.sample_rate,
            )
            arr = np.asarray(ts.value, dtype=np.float32)
            if len(arr) != cfg.n_samples:
                # GWOSC can return a sample more or less due to time-grid
                # rounding.  Centre-crop only; never pad fabricated data.
                offset = (len(arr) - cfg.n_samples) // 2
                arr = arr[offset:offset + cfg.n_samples]
            if len(arr) != cfg.n_samples or not np.all(np.isfinite(arr)):
                raise RuntimeError("returned data has an invalid length or non-finite values")
        except Exception as exc:
            warnings.warn(
                f"Skipping {event_name} for {cfg.detector}: unable to fetch "
                f"recorded event strain ({exc})"
            )
            continue

        events.append({
            "event_name": event_name,
            "data": arr,
            "gps_start": gps - half_window,
            "event_gps": gps,
        })
    return events


def estimate_psd(long_ts, cfg: Config):
    from pycbc.types import TimeSeries as PyCBCTimeSeries
    from pycbc.psd import welch, interpolate

    pycbc_ts = PyCBCTimeSeries(long_ts.value.astype(np.float64), delta_t=1.0 / cfg.sample_rate)
    seg_len = min(len(pycbc_ts), 4 * cfg.sample_rate)  # 4s segments for Welch
    seg_len = seg_len - (seg_len % 2)
    if seg_len < cfg.sample_rate:
        seg_len = len(pycbc_ts) - (len(pycbc_ts) % 2)
    psd = welch(pycbc_ts, seg_len=seg_len, seg_stride=seg_len // 2)
    psd = interpolate(psd, 1.0 / cfg.duration)
    return psd


# WAVEFORM INJECTION

def generate_waveform(template: TemplateParams, cfg: Config) -> np.ndarray:
    from pycbc.waveform import get_td_waveform

    hp, _ = get_td_waveform(
        approximant=cfg.approximant,
        mass1=template.mass1, mass2=template.mass2,
        delta_t=1.0 / cfg.sample_rate, f_lower=template.f_lower,
    )
    return physical_waveform_support(hp, cfg.sample_rate)


def place_and_scale_waveform(hp, psd, target_snr: float, cfg: Config,
                              rng: np.random.Generator) -> Tuple[np.ndarray, float, int]:

    from pycbc.filter import sigma
    from pycbc.types import TimeSeries as PyCBCTimeSeries
    from scipy.signal.windows import tukey

    hp_arr = np.asarray(hp.numpy() if hasattr(hp, "numpy") else hp, dtype=np.float64)
    n = cfg.n_samples

    if len(hp_arr) > int(cfg.max_waveform_fraction * n):
        keep = int(cfg.max_waveform_fraction * n)
        hp_arr = hp_arr[-keep:].copy()
        hp_arr *= tukey(len(hp_arr), alpha=0.1)
  
    max_start = n - len(hp_arr)
    start = int(rng.integers(0, max_start + 1))
    end = start + len(hp_arr)

    placed = np.zeros(n, dtype=np.float64)
    placed[start:end] = hp_arr
    peak_index = int(np.argmax(np.abs(placed)))

    placed_pycbc = PyCBCTimeSeries(placed, delta_t=1.0 / cfg.sample_rate)
    current_sigma = sigma(placed_pycbc, psd=psd, low_frequency_cutoff=25.0)

    if current_sigma <= 0 or not np.isfinite(current_sigma):
        raise RuntimeError("Non-finite optimal SNR; check PSD / waveform placement.")

    scale = target_snr / current_sigma
    placed_scaled = placed * scale
    actual_snr = target_snr  # by construction (linear scaling of matched-filter SNR)

    return placed_scaled.astype(np.float32), float(actual_snr), peak_index


def sample_snr(cfg: Config, rng: np.random.Generator) -> float:
    weights = [b[0] for b in cfg.snr_bins]
    idx = rng.choice(len(cfg.snr_bins), p=weights)
    _, lo, hi = cfg.snr_bins[idx]
    return float(rng.uniform(lo, hi))

In [ ]:
class IncrementalHDF5Writer:
    
    SCALAR_FIELDS = {
        "label": np.int8, "snr": np.float32,
        "mass1": np.float32, "mass2": np.float32,
        "f_lower": np.float32, "peak_index": np.int32,
    }
    STR_FIELDS = ["sample_id", "template_id", "noise_segment_id", "split"]

    def __init__(self, h5file: h5py.File, split: str, n_samples: int, chunk_size: int = 200):
        self.f = h5file
        self.split = split
        self.n_samples = n_samples
        self.grp = self.f.require_group(split)
        self.chunk_size = chunk_size
        self._n = 0
        self._init_datasets()

    def _init_datasets(self):
        if "strain" not in self.grp:
            self.grp.create_dataset(
                "strain", shape=(0, self.n_samples), maxshape=(None, self.n_samples),
                dtype=np.float32, chunks=(min(self.chunk_size, 64), self.n_samples),
                compression="gzip", compression_opts=4,
            )
            for name, dtype in self.SCALAR_FIELDS.items():
                self.grp.create_dataset(name, shape=(0,), maxshape=(None,), dtype=dtype,
                                         chunks=(self.chunk_size,))
            str_dt = h5py.string_dtype(encoding="utf-8")
            for name in self.STR_FIELDS:
                self.grp.create_dataset(name, shape=(0,), maxshape=(None,), dtype=str_dt,
                                         chunks=(self.chunk_size,))
        else:
            self._n = self.grp["strain"].shape[0]

    def append(self, strain: np.ndarray, record: Dict):
        i = self._n
        for name in ["strain"] + list(self.SCALAR_FIELDS.keys()) + self.STR_FIELDS:
            ds = self.grp[name]
            ds.resize(i + 1, axis=0)
        self.grp["strain"][i] = strain
        for name in self.SCALAR_FIELDS:
            self.grp[name][i] = record[name]
        for name in self.STR_FIELDS:
            self.grp[name][i] = record[name]
        self._n += 1

    def flush(self):
        self.f.flush()

#  MAIN PIPELINE
def make_sample_id(split: str, label: int, counter: int) -> str:
    return f"{split}_{label}_{counter:06d}"


def add_dataset_provenance(record: Dict, cfg: Config) -> Dict:
    return {
        **record,
        "detector": cfg.detector,
        "sample_rate": cfg.sample_rate,
        "duration": cfg.duration,
    }


def run_real_event_pipeline(cfg: Config):
    os.makedirs(cfg.output_dir, exist_ok=True)
    h5_path = os.path.join(cfg.output_dir, cfg.hdf5_filename)
    metadata_rows = []
    total_events = sum(len(v) for v in cfg.real_event_names.values())
    if total_events == 0:
        raise ValueError("real_event_names must contain at least one event.")

    print("Selecting science-quality off-source noise stretches from GWOSC...")
    cfg.noise_time_ranges, cfg.known_event_gps = auto_select_noise_time_ranges(
        detector=cfg.detector,
    )

    with h5py.File(h5_path, "w") as h5f:
        for split in ("train", "val", "test"):
            print(f"Fetching recorded GWOSC events for split='{split}'...")
            positives = fetch_real_event_segments(cfg, split)
            if not positives:
                raise RuntimeError(
                    f"No real event strain could be fetched for split={split}. "
                    "Check internet access, the detector, or real_event_names."
                )

            writer = IncrementalHDF5Writer(h5f, split, cfg.n_samples)
            sample_counter = 0
            for event in positives:
                record = {
                    "sample_id": make_sample_id(split, 1, sample_counter),
                    "label": 1,
                    "template_id": f"real:{event['event_name']}",
                    "noise_segment_id": f"{cfg.detector}_event_{event['event_name']}_{event['event_gps']:.3f}",
                    "split": split,
                    "snr": -1.0, 
                    "mass1": -1.0,
                    "mass2": -1.0,
                    "f_lower": -1.0,
                    "peak_index": cfg.n_samples // 2,
                }
                writer.append(event["data"], record)
                metadata_rows.append(add_dataset_provenance(
                    {**record, "gps_start": event["gps_start"]}, cfg
                ))
                sample_counter += 1

            print(f"Fetching off-source real noise for split='{split}'...")
            noise_pool = fetch_noise_segments(cfg, split)
            if not noise_pool:
                raise RuntimeError(f"No usable real-noise windows fetched for split={split}.")
            n_noise = max(1, round(cfg.n_noise_only_total * len(positives) / total_events))
            for noise in noise_pool[:n_noise]:
                record = {
                    "sample_id": make_sample_id(split, 0, sample_counter),
                    "label": 0,
                    "template_id": "None",
                    "noise_segment_id": noise["noise_segment_id"],
                    "split": split,
                    "snr": 0.0,
                    "mass1": -1.0,
                    "mass2": -1.0,
                    "f_lower": -1.0,
                    "peak_index": -1,
                }
                writer.append(noise["data"], record)
                metadata_rows.append(add_dataset_provenance(
                    {**record, "gps_start": noise["gps_start"]}, cfg
                ))
                sample_counter += 1
            writer.flush()
            print(f"  {split}: {len(positives)} real events + {n_noise} real-noise windows")

        h5f.attrs["detector"] = cfg.detector
        h5f.attrs["sample_rate"] = cfg.sample_rate
        h5f.attrs["duration"] = cfg.duration
        h5f.attrs["n_samples"] = cfg.n_samples
        h5f.attrs["signal_source"] = "GWOSC catalogue event-centred strain"
        h5f.attrs["whitened"] = False
        h5f.attrs["generated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    meta_path = os.path.join(cfg.output_dir, cfg.metadata_filename)
    meta_df = pd.DataFrame(metadata_rows)
    meta_df.to_csv(meta_path, index=False)
    print(f"Done. Raw HDF5: {h5_path} | metadata: {meta_path}")
    return h5_path, meta_path, meta_df


def run_pipeline(cfg: Config, mode: str = "test"):
    """
    mode="test": tiny smoke-test (2 templates/split, 3 examples each, ~30 noise-only)
    mode="full": full dataset as specified in cfg
    """
    rng = np.random.default_rng(cfg.random_seed)
    diagnostic_done = False
    os.makedirs(cfg.output_dir, exist_ok=True)
    h5_path = os.path.join(cfg.output_dir, cfg.hdf5_filename)

    if mode == "test":
        test_cfg = Config(
            n_templates=6, n_train_templates=4, n_val_templates=1, n_test_templates=1,
            n_examples_per_template=3, n_noise_only_total=12,
            output_dir=cfg.output_dir,
            hdf5_filename="gw_dataset_TEST.h5",
            noise_time_ranges=cfg.noise_time_ranges,
            known_event_gps=cfg.known_event_gps,
        )
        cfg = test_cfg
        h5_path = os.path.join(cfg.output_dir, cfg.hdf5_filename)

    print("[0/6] Selecting verified science-quality noise segments from GWOSC...")

    auto_noise_ranges, auto_event_gps = auto_select_noise_time_ranges(
    run_start=1238166018,
    run_end=1253977218,
    detector=cfg.detector,
    train_hours=2.0,
    val_hours=0.75,
    test_hours=0.75,
    min_segment_seconds=600,
    )

    cfg.noise_time_ranges = auto_noise_ranges
    cfg.known_event_gps = auto_event_gps

    print("Selected noise ranges:")
    for split, ranges in cfg.noise_time_ranges.items():
        print(f"  {split}: {ranges}")

    print(f"Excluded real GW events: {len(cfg.known_event_gps)}")
    print(f"[1/6] Generating {cfg.n_templates} template parameter sets...")
    templates = generate_template_bank(cfg, rng)
    print(f"[2/6] Fitting f_lower per template for the 1-second analysis window...")
    templates = [fit_f_lower(t, cfg) for t in templates]

    print(f"[3/6] Splitting templates (template-disjoint: "
          f"{cfg.n_train_templates}/{cfg.n_val_templates}/{cfg.n_test_templates})...")
    splits = split_templates(templates, cfg, rng)

    metadata_rows = []
    used_noise_ids_global = set()

    with h5py.File(h5_path, "w") as h5f:
        for split_name, split_templates_list in splits.items():
            print(f"[4/6] Fetching real noise for split='{split_name}'...")
            noise_pool = fetch_noise_segments(cfg, split_name)
            if len(noise_pool) == 0:
                raise RuntimeError(
                    f"No usable noise segments fetched for split={split_name}. "
                    f"Check network access / GWOSC availability / time ranges."
                )
            rng.shuffle(noise_pool)
            noise_cursor = 0

            writer = IncrementalHDF5Writer(h5f, split_name, cfg.n_samples)
            sample_counter = 0

            # ---- positive (GW + noise) examples ----
            print(f"    Injecting waveforms for {len(split_templates_list)} templates "
                  f"x {cfg.n_examples_per_template} examples...")
            psd_cache: Dict[int, object] = {}  # cache PSD per fetched noise chunk (id(raw_ts) -> psd)
            injections_done = 0
            total_injections = len(split_templates_list) * cfg.n_examples_per_template
            progress_every = max(1, total_injections // 10)

            for template in split_templates_list:
                hp = generate_waveform(template, cfg)
                for _ in range(cfg.n_examples_per_template):
                    if noise_cursor >= len(noise_pool):
                        rng.shuffle(noise_pool)
                        noise_cursor = 0
                    noise_entry = noise_pool[noise_cursor]
                    noise_cursor += 1

                    raw_ts_key = id(noise_entry["raw_ts"])
                    if raw_ts_key not in psd_cache:
                        psd_cache[raw_ts_key] = estimate_psd(noise_entry["raw_ts"], cfg)
                    psd = psd_cache[raw_ts_key]
                    target_snr = sample_snr(cfg, rng)

                    try:
                        placed_wave, actual_snr, peak_idx = place_and_scale_waveform(
                            hp, psd, target_snr, cfg, rng
                        )
                    except Exception as e:
                        warnings.warn(f"Injection failed for {template.template_id}: {e}")
                        continue
                    
                    # NORMAL DATASET GENERATION
                    
                    injected = noise_entry["data"].astype(np.float32) + placed_wave
                    
                    if not np.all(np.isfinite(injected)):
                        warnings.warn(
                            f"Non-finite sample skipped ({template.template_id})"
                        )
                        continue

                    sample_id = make_sample_id(split_name, 1, sample_counter)
                    record = {
                        "sample_id": sample_id, "label": 1,
                        "template_id": template.template_id,
                        "noise_segment_id": noise_entry["noise_segment_id"],
                        "split": split_name, "snr": actual_snr,
                        "mass1": template.mass1, "mass2": template.mass2,
                        "f_lower": template.f_lower,
                        "peak_index": peak_idx,
                    }
                    writer.append(injected, record)
                    metadata_rows.append(add_dataset_provenance(record, cfg))
                    used_noise_ids_global.add(noise_entry["noise_segment_id"])
                    sample_counter += 1
                    injections_done += 1
                    if injections_done % progress_every == 0:
                        print(f"      ...{injections_done}/{total_injections} injections done")

            # ---- noise-only examples ----
            n_noise_only_split = int(round(
                cfg.n_noise_only_total * len(split_templates_list) / cfg.n_templates
            ))
            print(f"    Adding {n_noise_only_split} noise-only examples...")
            for _ in range(n_noise_only_split):
                if noise_cursor >= len(noise_pool):
                    rng.shuffle(noise_pool)
                    noise_cursor = 0
                noise_entry = noise_pool[noise_cursor]
                noise_cursor += 1

                data = noise_entry["data"].astype(np.float32)
                if not np.all(np.isfinite(data)):
                    continue

                sample_id = make_sample_id(split_name, 0, sample_counter)
                record = {
                    "sample_id": sample_id, "label": 0,
                    "template_id": "None",
                    "noise_segment_id": noise_entry["noise_segment_id"],
                    "split": split_name, "snr": 0.0,
                    "mass1": -1.0, "mass2": -1.0,
                    "f_lower": -1.0,
                    "peak_index": -1,
                }
                writer.append(data, record)
                metadata_rows.append(add_dataset_provenance(record, cfg))
                sample_counter += 1

            writer.flush()
            print(f"    split='{split_name}' done: {sample_counter} total examples.")

        # store run-level attributes for reproducibility
        h5f.attrs["detector"] = cfg.detector
        h5f.attrs["sample_rate"] = cfg.sample_rate
        h5f.attrs["duration"] = cfg.duration
        h5f.attrs["n_samples"] = cfg.n_samples
        h5f.attrs["whitened"] = False
        h5f.attrs["generated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    print(f"[5/6] Writing metadata CSV...")
    meta_df = pd.DataFrame(metadata_rows)
    meta_path = os.path.join(cfg.output_dir, cfg.metadata_filename)
    meta_df.to_csv(meta_path, index=False)

    print(f"[6/6] Done. HDF5: {h5_path}  |  metadata: {meta_path}")
    return h5_path, meta_path, splits, meta_df

# ENTRY POINT

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--mode",
        choices=["real-events", "test", "full"],
        default="full",
        help=("full creates 80 PyCBC templates injected into real GWOSC/LIGO "
              "noise; test creates 6 templates; real-events is optional"),
    )
    args, _unknown = parser.parse_known_args()

    cfg = Config()
    if args.mode == "real-events":
        run_real_event_pipeline(cfg)
    else:
        run_pipeline(cfg, mode=args.mode)